### Script: rebuild_cases.py

In [ ]:
import re
import json
import logging
import pandas as pd
from pathlib import Path
from tqdm import tqdm

CLEAN_DIR  = Path("data/processed/clean")
OUTPUT_DIR = Path("data/processed")

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
log = logging.getLogger(__name__)



def clean_text(text: str) -> str:
    """
    Preprocessing teks putusan yang lebih komprehensif:
    1. Lowercase
    2. Hapus disclaimer Mahkamah Agung (seluruh blok)
    3. Perbaiki 'p u t u s a n' → 'putusan' (spasi antar huruf OCR)
    4. Perbaiki kata menyatu akibat newline PDF (e.g. 'malangdemi' → 'malang demi')
    5. Hapus 'halaman N' berulang
    6. Normalisasi spasi
    """
    text = text.lower().strip()

    text = re.sub(
        r"disclaimer.*?ext\.318\)",
        "",
        text,
        flags=re.DOTALL | re.IGNORECASE
    )

    text = re.sub(r"\bhalaman\s*\d+\b", " ", text, flags=re.IGNORECASE)
    text = re.sub(r"\bhal\.\s*\d+\b", " ", text, flags=re.IGNORECASE)
    text = re.sub(r"(?<!\w)-\s*\d+\s*-(?!\w)", " ", text)

    def fix_spaced_chars(m):
        return m.group(0).replace(" ", "")
    text = re.sub(r"(?<!\w)([a-z] ){3,}[a-z](?!\w)", fix_spaced_chars, text)

    kata_kunci = [
        "demi", "pengadilan", "negeri", "yang", "mengadili", "perkara",
        "pidana", "dengan", "acara", "pemeriksaan", "biasa", "dalam",
        "tingkat", "pertama", "menjatuhkan", "putusan", "sebagai",
        "berikut", "nama", "tempat", "lahir", "umur", "tanggal",
        "jenis", "kelamin", "kebangsaan", "tinggal", "agama",
        "pekerjaan", "terdakwa", "menimbang", "bahwa", "mengadili",
        "menyatakan", "menetapkan", "memutuskan", "berdasarkan",
        "ketuhanan", "maha", "esa", "nomor", "terhadap", "pasal",
        "jaksa", "penuntut", "umum", "hakim", "majelis", "panitera",
        "saksi", "korban", "terbukti", "bersalah", "pidana", "penjara",
        "bulan", "tahun", "denda", "biaya", "perkara",
    ]
    pattern_menyatu = r"([a-z])(" + "|".join(kata_kunci) + r")(?=[a-z\s])"
    text = re.sub(pattern_menyatu, r"\1 \2", text)

    text = re.sub(r"kepaniteraan@mahkamahagung\.go\.id", "", text)
    text = re.sub(r"telp\s*:\s*021[-\d\s\.]+\(ext\.\d+\)", "", text)

    text = re.sub(r"\s+", " ", text)

    return text.strip()



def ekstrak_no_perkara(text):
    patterns = [
        r"\d+\s*/\s*pid\.b\s*/\s*\d{4}\s*/\s*pn\s*[\w\.]+",
        r"\d+\s*/\s*pid\.sus\s*/\s*\d{4}\s*/\s*pn\s*[\w\.]+",
        r"nomor\s*[:\s]+(\d+[/\-][\w\.]+[/\-]\d{4}[/\-][\w\.]+)",
    ]
    for pat in patterns:
        m = re.search(pat, text, re.IGNORECASE)
        if m:
            return m.group(0).strip()
    return ""


def ekstrak_tanggal(text):
    bulan_map = {
        "januari": "01", "februari": "02", "maret": "03", "april": "04",
        "mei": "05", "juni": "06", "juli": "07", "agustus": "08",
        "september": "09", "oktober": "10", "november": "11", "desember": "12"
    }
    pat = r"(\d{1,2})\s+(januari|februari|maret|april|mei|juni|juli|agustus|september|oktober|november|desember)\s+(\d{4})"
    m = re.search(pat, text, re.IGNORECASE)
    if m:
        hari, bln, thn = m.group(1), m.group(2).lower(), m.group(3)
        return f"{thn}-{bulan_map.get(bln, '00')}-{int(hari):02d}"
    return ""


def ekstrak_terdakwa(text):
    patterns = [
        r"terdakwa\s*[:\s]+([A-Z][a-zA-Z\s]+?)(?:\s*,|\s*alias|\s*bin|\s*binti|\n)",
        r"nama\s*lengkap\s*[:\s]+([A-Z][a-zA-Z\s]+?)(?:\s*;|\s*,|\n)",
        r"nama\s*[:\s]+([A-Z][a-zA-Z\s\.]+?)(?:\s*;|\s*,|\s*\n)",
    ]
    for pat in patterns:
        m = re.search(pat, text, re.IGNORECASE)
        if m:
            nama = m.group(1).strip()
            if 3 < len(nama) < 60:
                return nama.title()
    return ""


def ekstrak_pasal_dakwaan(text):
    patterns = [
        r"pasal\s+362\s+kuhp",
        r"pasal\s+363\s+(?:ayat\s+\(\d\)\s+)?kuhp",
        r"pasal\s+364\s+kuhp",
        r"pasal\s+365\s+kuhp",
    ]
    found = []
    for pat in patterns:
        matches = re.findall(pat, text, re.IGNORECASE)
        found.extend(matches)
    return "; ".join(set(found)) if found else ""


def ekstrak_amar_putusan(text):
    m = re.search(
        r"m\s*e\s*n\s*g\s*a\s*d\s*i\s*l\s*i(.{50,800}?)(?=menimbang|menetapkan|demikian|$)",
        text, re.IGNORECASE | re.DOTALL
    )
    if m:
        blok = m.group(1).lower().replace('\n', ' ')
        blok = re.sub(r'\s+', ' ', blok).strip()
        if re.search(r"terbukti.*bersalah|menyatakan.*terdakwa.*bersalah", blok):
            return "bersalah"
        if re.search(r"tidak terbukti|membebaskan terdakwa", blok):
            return "bebas"
        if re.search(r"melepaskan terdakwa|lepas dari", blok):
            return "lepas"
        return blok[:200].strip()
    return ""


def ekstrak_vonis_bulan(text):
    text = text.replace("pidanapenjara", "pidana penjara")
    
    gabungan_m = re.search(r"penjara\s+selama\s+(\d+)\s*\([\w\s]+\)\s*tahun\s+dan\s+(\d+)\s*\([\w\s]+\)\s*bulan", text, re.IGNORECASE)
    if gabungan_m:
        return (int(gabungan_m.group(1)) * 12) + int(gabungan_m.group(2))
        
    tahun_m = re.search(r"penjara\s+selama\s+(\d+)\s*\([\w\s]+\)\s*tahun", text, re.IGNORECASE)
    bulan_m = re.search(r"penjara\s+selama\s+(\d+)\s*\([\w\s]+\)\s*bulan", text, re.IGNORECASE)
    
    if not tahun_m and not bulan_m:
        tahun_m = re.search(r"penjara\s+(\d+)\s+tahun", text, re.IGNORECASE)
        bulan_m = re.search(r"penjara\s+(\d+)\s+bulan", text, re.IGNORECASE)
        
    total = 0
    if tahun_m:
        total += int(tahun_m.group(1)) * 12
    if bulan_m:
        total += int(bulan_m.group(1))
    return total


def ekstrak_ringkasan_fakta(text):
    patterns = [
        r"menimbang.*?bahwa(.{100,800}?)(?=menimbang|mempertimbangkan|mengenai|$)",
        r"bahwa terdakwa(.{100,600}?)(?=bahwa|menimbang|$)",
    ]
    for pat in patterns:
        m = re.search(pat, text, re.IGNORECASE | re.DOTALL)
        if m:
            snippet = re.sub(r"\s+", " ", m.group(1).replace("\n", " ").strip())
            return snippet[:600]
    return ""


def ekstrak_argumen_hukum(text):
    patterns = [
        r"mempertimbangkan(.{100,800}?)(?=menimbang|mengadili|menetapkan|$)",
        r"pertimbangan hukum(.{100,600}?)(?=mengadili|menetapkan|$)",
    ]
    for pat in patterns:
        m = re.search(pat, text, re.IGNORECASE | re.DOTALL)
        if m:
            snippet = re.sub(r"\s+", " ", m.group(1).replace("\n", " ").strip())
            return snippet[:600]
    return ""



def main():
    txt_files = sorted(CLEAN_DIR.rglob("*.txt"))
    log.info(f"Ditemukan {len(txt_files)} file txt")

    rows = []
    stats = {"teks_penuh_chars": [], "teks_clean_chars": []}

    for txt_path in tqdm(txt_files, desc="Rebuild cases"):
        label_folder = txt_path.parent.name
        raw_text = txt_path.read_text(encoding="utf-8", errors="ignore")
        cleaned  = clean_text(raw_text)

        stats["teks_penuh_chars"].append(len(cleaned))
        stats["teks_clean_chars"].append(len(raw_text))

        row = {
            "case_id":         txt_path.stem,
            "filename":        txt_path.name,
            "label_pasal":     label_folder,
            "no_perkara":      ekstrak_no_perkara(cleaned),
            "tanggal":         ekstrak_tanggal(cleaned),
            "pengadilan":      "PN Malang",
            "terdakwa":        ekstrak_terdakwa(cleaned),
            "pasal_dakwaan":   ekstrak_pasal_dakwaan(cleaned),
            "amar_putusan":    ekstrak_amar_putusan(cleaned),
            "vonis_bulan":     ekstrak_vonis_bulan(cleaned),
            "ringkasan_fakta": ekstrak_ringkasan_fakta(cleaned),
            "argumen_hukum":   ekstrak_argumen_hukum(cleaned),
            "word_count":      len(cleaned.split()),
            "text_full":       cleaned,
        }
        rows.append(row)

    json_path = OUTPUT_DIR / "cases.json"
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(rows, f, ensure_ascii=False, indent=2)
    log.info(f"cases.json disimpan: {json_path}")

    df = pd.DataFrame(rows)
    csv_path = OUTPUT_DIR / "cases.csv"
    df.to_csv(csv_path, index=False, encoding="utf-8-sig")
    log.info(f"cases.csv disimpan: {csv_path}")

    avg_chars = sum(stats["teks_penuh_chars"]) / len(stats["teks_penuh_chars"])
    min_chars = min(stats["teks_penuh_chars"])
    max_chars = max(stats["teks_penuh_chars"])
    log.info(f"\n{'='*50}")
    log.info(f"STATISTIK TEKS BARU:")
    log.info(f"  Rata-rata panjang teks: {avg_chars:.0f} karakter")
    log.info(f"  Min: {min_chars} | Max: {max_chars}")
    log.info(f"  Total kasus: {len(rows)}")
    log.info(f"{'='*50}")

    log.info("\nCONTOH TEKS SETELAH PERBAIKAN (500 char pertama, kasus pertama):")
    log.info(rows[0]["text_full"][:500])


if __name__ == "__main__":
    main()



### Script: rebuild_models.py

In [ ]:
"""
rebuild_models.py
Rebuild TF-IDF vectorizer + IndoBERT embeddings dari cases.json yang baru.
Jalankan setelah rebuild_cases.py.
"""

import json
import pickle
import logging
import numpy as np
from pathlib import Path
from tqdm import tqdm

DATA_DIR  = Path("data/processed")
MODEL_DIR = Path("models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
log = logging.getLogger(__name__)


def load_cases():
    with open(DATA_DIR / "cases.json", encoding="utf-8") as f:
        return json.load(f)



def rebuild_tfidf(cases):
    from sklearn.feature_extraction.text import TfidfVectorizer

    log.info("Rebuild TF-IDF vectorizer...")
    texts = [c["text_full"] for c in cases]

    vectorizer = TfidfVectorizer(
        min_df=1,
        max_df=0.95,
        ngram_range=(1, 2),
        sublinear_tf=True,
    )
    vectorizer.fit(texts)

    pkl_path = MODEL_DIR / "tfidf_vectorizer.pkl"
    with open(pkl_path, "wb") as f:
        pickle.dump(vectorizer, f)

    log.info(f"TF-IDF disimpan: {pkl_path}")
    log.info(f"  Vocab size: {len(vectorizer.vocabulary_)}")
    return vectorizer



def get_embedding(text, tokenizer, model, device):
    import torch

    tokens = tokenizer(text, add_special_tokens=False, return_tensors="pt")["input_ids"][0]
    chunk_size = 510
    chunks = [tokens[i:i+chunk_size] for i in range(0, len(tokens), chunk_size)]

    chunk_embs = []
    for chunk in chunks:
        input_ids = torch.cat([
            torch.tensor([tokenizer.cls_token_id]),
            chunk,
            torch.tensor([tokenizer.sep_token_id])
        ]).unsqueeze(0).to(device)
        attention_mask = torch.ones_like(input_ids).to(device)

        with torch.no_grad():
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)

        mask = attention_mask.unsqueeze(-1).float()
        emb = (outputs.last_hidden_state * mask).sum(1) / mask.sum(1)
        chunk_embs.append(emb)

    if chunk_embs:
        return torch.stack(chunk_embs).mean(dim=0).cpu().numpy()[0]
    else:
        return np.zeros(model.config.hidden_size)


def rebuild_indobert(cases):
    import torch
    from transformers import AutoModel, AutoTokenizer

    log.info("Rebuild IndoBERT embeddings...")
    MODEL_NAME = "indobenchmark/indobert-base-p1"
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    log.info(f"  Device: {device}")

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModel.from_pretrained(MODEL_NAME).to(device)
    model.eval()

    embeddings = []
    for case in tqdm(cases, desc="IndoBERT embeddings"):
        emb = get_embedding(case["text_full"], tokenizer, model, device)
        embeddings.append(emb)

    embeddings_array = np.array(embeddings)
    npy_path = MODEL_DIR / "indobert_embeddings.npy"
    np.save(npy_path, embeddings_array)
    log.info(f"Embeddings disimpan: {npy_path}")
    log.info(f"  Shape: {embeddings_array.shape}")
    return embeddings_array



if __name__ == "__main__":
    cases = load_cases()
    log.info(f"Loaded {len(cases)} kasus")

    rebuild_tfidf(cases)

    log.info("\nMulai rebuild IndoBERT embeddings (ini butuh waktu)...")
    rebuild_indobert(cases)

    log.info("\nSelesai! Restart aplikasi Streamlit untuk menggunakan model baru.")



### Script: diagnose_text.py

In [ ]:
import json
import re

cases = json.load(open('data/processed/cases.json', encoding='utf-8'))

c = cases[0]
print(f"Panjang text_full: {len(c['text_full'])}")
print()
print("=== 500 karakter TERAKHIR ===")
print(repr(c['text_full'][-500:]))

print()
print("=== POLA TEKS BERMASALAH ===")
text = c['text_full']
pattern_spaced = re.findall(r'(?:[a-z] ){3,}[a-z]', text)
print("Teks dengan spasi antar huruf (OCR artifact):")
for p in pattern_spaced[:5]:
    print(f"  '{p}'")

print()
pattern_menyatu = re.findall(r'[a-z]{3,}[a-z][A-Z][a-z]', text)
print("Kata menyatu tanpa spasi:")
for p in pattern_menyatu[:5]:
    print(f"  '{p}'")

print()
pattern_menyatu2 = re.findall(r'(?<=[a-z])(?=[A-Z])', text)
print(f"Jumlah kemungkinan kata menyatu (camelCase): {len(pattern_menyatu2)}")

print()
idx = text.find('p u t u s a n')
if idx == -1:
    idx = text.find('putusan')
if idx >= 0:
    print(f"Teks sekitar kata 'putusan' (posisi {idx}):")
    print(repr(text[max(0, idx-50):idx+200]))



### Script: test_clean.py

In [ ]:
import re
from collections import Counter

raw_sample = open(r"data\processed\clean\pasal_362\case_pasal_362_001.txt", encoding="utf-8").read()

def remove_page_boundaries(raw_text):
    lines = raw_text.split("\n")
    content_lines = []
    in_disclaimer = False
    for line in lines:
        stripped = line.strip()
        if not stripped:
            continue
        if stripped.lower() == "disclaimer":
            in_disclaimer = True
            continue
        if in_disclaimer:
            if "ext.318)" in stripped.lower():
                in_disclaimer = False
            continue
        if re.match(r"^halaman\s+\d+$", stripped, re.IGNORECASE):
            continue
        stripped = re.sub(
            r"\s*putusan\s+nomor\s+\d+/pid\.\w+/\d{4}/pn\s+\w+\s*$",
            "", stripped, flags=re.IGNORECASE
        )
        if stripped:
            content_lines.append(stripped)
    return " ".join(content_lines)

INDO_COMMON_WORDS = {
    "ada", "adalah", "adanya", "agar", "akan", "akibat", "akhir",
    "alasan", "amar", "anak", "antara", "apa", "apabila",
    "atas", "atau", "ayat",
    "bagi", "bagian", "bahwa", "baik", "barang", "baru",
    "batu", "beberapa", "belum", "benar", "berapa",
    "berdasarkan", "berdiri", "berikut", "berkas", "bermaksud",
    "bersalah", "bersama", "berupa", "besar", "biasa", "biaya",
    "bila", "bisa", "boleh", "buah", "bukan", "bulan",
    "cara", "cukup",
    "dahulu", "dalam", "dan", "dapat", "dari", "datang",
    "dengan", "demi", "demikian", "demikianlah",
    "denda", "depan", "desa", "dimana", "dimaksud",
    "diatur", "diancam", "didakwa", "didakwakan", "dijatuhkan",
    "dinyatakan", "diperoleh", "dipersidangan",
    "duduk", "dusun",
    "empat", "enam",
    "fakta",
    "hak", "hal", "halaman", "hanya", "hari", "hakim",
    "harus", "haruslah", "hasil", "hendak", "hukum", "hukuman",
    "ini", "islam", "itu",
    "jadi", "jalan", "jaksa", "jam", "jenis",
    "jika", "juga", "jumlah",
    "kabupaten", "kalau", "kami", "karena", "karenanya",
    "kasus", "kata", "keadilan", "kebangsaan",
    "kecamatan", "kedua", "kelamin",
    "kelurahan", "keluarga", "kemudian", "kepada",
    "kepadanya", "kepentingan", "keperluan",
    "kerja", "kerugian", "kesadaran",
    "keterangan", "ketentuan", "ketiga", "ketika",
    "ketua", "ketuhanan",
    "kewajiban", "khususnya",
    "korban", "kota", "kuasa", "kunci", "kurang",
    "lagi", "lahir", "lain", "lalu", "lama",
    "langsung", "lebih", "lengkap", "lima",
    "maha", "maka", "maksud", "malang",
    "mampu", "mana", "masa", "masing",
    "masih", "masuk", "masyarakat", "maupun",
    "majelis", "meja",
    "melakukan", "melanggar", "melawan",
    "melepaskan", "memang", "membebaskan",
    "membawa", "membayar", "memberi", "memberikan",
    "memenuhi", "memiliki",
    "memohon", "mempertimbangkan", "memperhatikan",
    "memutuskan",
    "mendapat", "mendekati", "mendorong",
    "mendengar", "menerangkan",
    "mengadili", "mengajukan", "mengakui", "mengalami",
    "mengambil", "mengenai", "mengetahui",
    "menggunakan", "mengingat", "menguasai",
    "menimbang", "menikmati", "menjadi",
    "menjatuhkan", "menuju",
    "menurut", "menyatakan", "menyesal", "menyesali",
    "meresahkan", "merupakan", "meskipun",
    "milik", "miliki", "motor", "mulai",
    "nama", "namun", "negeri", "nomor",
    "orang", "oleh",
    "pada", "paling", "panitera", "para", "parkir",
    "pasal", "pekerjaan",
    "pelaku", "pemaaf", "pembenar",
    "pembacaan", "pemeriksaan",
    "pencurian", "pendapat", "penetapan",
    "pengganti", "pengetahuan",
    "penjara", "penjeraan",
    "penuntut", "penunjukan",
    "penyesuaian", "penyidik",
    "peradilan", "peraturan",
    "perbuatan", "perbuatannya",
    "perkara", "perlu", "pernah", "persidangan",
    "pertama", "pertimbangan", "perundang",
    "pidana", "pihak", "pokoknya",
    "pula", "pukul", "putusan",
    "rasa", "rekaman", "republik", "resto",
    "ringan", "rupiah",
    "saat", "saja", "saksi", "salah",
    "sama", "sampai", "sangat", "sarana",
    "satu", "sebagai", "sebagaimana",
    "sebagian", "sebelum", "sebelumnya",
    "secara", "sedang", "sedangkan",
    "sehingga", "sejak", "sejumlah", "sekira", "sekitar",
    "selain", "selaku", "selama", "selanjutnya",
    "seluruh", "seluruhnya",
    "semua", "sendiri", "sependapat",
    "sepeda", "sepengetahuan",
    "seperti", "sepatutnya",
    "serta", "sesuai", "sesuatu",
    "setelah", "setiap", "setimpal",
    "sifat", "sopan",
    "suatu", "sudah", "sumpah", "supaya", "surat",
    "tahun", "tanggal", "tanpa", "telah",
    "tempat", "tentang", "tentunya",
    "terbukti", "terhadap",
    "terdakwa", "terlebih", "termasuk", "ternyata",
    "tersebut", "terungkap",
    "tetap", "tetapi",
    "tidak", "tindak", "tinggal",
    "tujuan", "tulang", "tuntutan",
    "tujuh", "tiga", "tingkat",
    "umum", "umur", "undang",
    "unsur", "untuk", "utama",
    "wajib", "waktu", "walaupun", "warna", "wib",
}

def split_merged_words(text, word_set, min_token_len=12, min_part_len=3):
    tokens = text.split()
    result = []
    for token in tokens:
        alpha_only = re.sub(r"[^a-z]", "", token)
        if len(alpha_only) <= min_token_len or alpha_only in word_set:
            result.append(token)
            continue
        best_split = None
        best_score = -1
        for i in range(min_part_len, len(alpha_only) - min_part_len + 1):
            left = alpha_only[:i]
            right = alpha_only[i:]
            if left in word_set and right in word_set:
                score = len(left) + len(right) + min(len(left), len(right))
                if score > best_score:
                    best_score = score
                    best_split = (left, right)
        if best_split:
            result.append(best_split[0])
            sub = split_merged_words(best_split[1], word_set, min_token_len, min_part_len)
            result.append(sub)
        else:
            result.append(token)
    return " ".join(result)

pass1 = remove_page_boundaries(raw_sample).lower()
pass1 = re.sub(r"\s+", " ", pass1).strip()

print("=== RAW (200 char) ===")
print(raw_sample[:200])
print()
print("=== PASS 1: Remove Boundaries (200 char) ===")
print(pass1[:200])
print()
print(f"RAW: {len(raw_sample)} chars")
print(f"PASS1: {len(pass1)} chars")
pct = (1 - len(pass1) / len(raw_sample)) * 100
print(f"Noise removed: {len(raw_sample) - len(pass1)} chars ({pct:.1f}%)")
print()

words_before = pass1.split()
long_before = [w for w in words_before if len(w) > 15 and w.isalpha()]
print(f"Long tokens (>15 char) BEFORE split: {len(long_before)}")
for t in long_before[:15]:
    print(f"  {t}")
print()

pass2 = split_merged_words(pass1, INDO_COMMON_WORDS)
pass2 = re.sub(r"\s+", " ", pass2).strip()

words_after = pass2.split()
long_after = [w for w in words_after if len(w) > 15 and w.isalpha()]
print(f"Long tokens (>15 char) AFTER split: {len(long_after)}")
for t in long_after[:15]:
    print(f"  {t}")
print()

print("=== FINAL CLEAN (300 char) ===")
print(pass2[:300])
print()

has_disc = "disclaimer" in pass2
has_ext = "ext.318" in pass2
has_email = "kepaniteraan" in pass2
has_page = bool(re.search(r"halaman \d+", pass2))
print(f"Contains 'disclaimer': {has_disc}")
print(f"Contains 'ext.318': {has_ext}")
print(f"Contains 'kepaniteraan': {has_email}")
print(f"Contains 'halaman N': {has_page}")



### Script: get_sample.py

In [ ]:
import json
import random

with open('data/processed/cases.json', encoding='utf-8') as f:
    cases = json.load(f)

cases_363 = [x for x in cases if x['label_pasal'] == 'pasal_363']
cases_362 = [x for x in cases if x['label_pasal'] == 'pasal_362']

random.seed(42)
sample_363 = random.sample(cases_363, 2)
sample_362 = random.sample(cases_362, 2)

samples = sample_363 + sample_362

for i, c in enumerate(samples, 1):
    print(f"--- CONTOH {i} ---")
    print(f"Prediksi Seharusnya:")
    print(f"Pasal: {c['label_pasal']}")
    print(f"Vonis: {c['vonis_bulan']} bulan\n")
    print(f"Teks Putusan:\n{c['text_full'][:1000]}...\n")

